# Detectability limits of scaling laws: example calculation

Worked example of the subgroup analysis in the paper, on the mammalian metabolic data (Fig. 2(b), SM Sec. C.2). Everything is in this notebook: load PanTHERIA, fit a common exponent with order-specific prefactors, fit each order on its own, and ask whether the orders differ in prefactor or in exponent by more than the resolution limits allow.

The other datasets used in the paper (BEA metropolitan panels, road-length panel) are in `data/` as well (see README).

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2, norm

Z = 2.0            # two standard errors (KL = 2)
MIN_SPECIES = 20   # an order needs this many species
MIN_FAMILIES = 5   # and this many families

## Data

PanTHERIA gives basal metabolic rate (mL O2/hr) and the body mass measured alongside it. We keep species with both, then the orders with at least 20 species and 5 families.

In [ ]:
df = pd.read_csv('data/allometry/PanTHERIA_1-0_WR05_Aug2008.txt', sep='\t',
                 na_values=['-999', '-999.00'])
cols = {'order': 'MSW05_Order', 'family': 'MSW05_Family',
        'mass': '5-2_BasalMetRateMass_g', 'bmr': '18-1_BasalMetRate_mLO2hr'}
df = df[list(cols.values())].dropna()
df.columns = list(cols.keys())
print(f'{len(df)} species with both mass and BMR')

counts = df.groupby('order').size()
fams = df.groupby('order')['family'].nunique()
keep = [o for o in counts.index if counts[o] >= MIN_SPECIES and fams[o] >= MIN_FAMILIES]
df = df[df['order'].isin(keep)].reset_index(drop=True)
orders = sorted(keep, key=lambda o: -counts[o])
x = np.log(df['mass'].to_numpy(float))    # log body mass
y = np.log(df['bmr'].to_numpy(float))     # log metabolic rate
order = df['order'].to_numpy()
family = df['family'].to_numpy()
print(f'{len(df)} species in {len(orders)} orders:', {o: int(counts[o]) for o in orders})

## Common exponent with order-specific prefactors

Each order plays the role of a system $i$ in Eq. (1), $y = c_i + \beta\, x + \xi$, and its species are the observations. Demeaning within each order removes the prefactors and gives the common slope; each order's offset $\hat c_k$ is then read off at that slope.

In [ ]:
xd, yd = x.copy(), y.copy()
for o in orders:
    msk = order == o
    xd[msk] -= x[msk].mean()
    yd[msk] -= y[msk].mean()
ss_w = (xd ** 2).sum()
beta_common = (xd * yd).sum() / ss_w
resid = yd - beta_common * xd
sigma = np.sqrt((resid ** 2).sum() / (len(x) - len(orders) - 1))
se_common = sigma / np.sqrt(ss_w)
print(f'common exponent {beta_common:.3f} +/- {Z * se_common:.3f}  (residual sd {sigma:.2f})')

## Each order on its own

Per-order slope $\hat\beta_k$ by least squares within the order, and offset $\hat c_k$ at the common slope. Two error estimates for each: treating the species residuals as independent (variance $\hat\sigma^2/\mathrm{ss}_k$ for the slope, $\hat\sigma^2/n_k$ for the offset), and clustering the residuals by family, since related species share history.

In [ ]:
rows = []
for o in orders:
    msk = order == o
    xk, yk, fk = x[msk], y[msk], family[msk]
    n = msk.sum()
    xc = xk - xk.mean()
    ss = (xc ** 2).sum()
    slope = (xc * (yk - yk.mean())).sum() / ss
    offset = yk.mean() - beta_common * xk.mean()
    # independent residuals
    v_slope = sigma ** 2 / ss
    v_offset = sigma ** 2 / n
    # clustered by family (CR1 small-sample factor)
    e_slope = (yk - yk.mean()) - slope * xc
    u_off = yk - beta_common * xk - offset
    fam_list = np.unique(fk)
    F = len(fam_list)
    s_sl = np.array([(xc[fk == f] * e_slope[fk == f]).sum() for f in fam_list])
    s_of = np.array([u_off[fk == f].sum() for f in fam_list])
    cr1 = F / (F - 1)
    v_slope_fam = cr1 * (s_sl ** 2).sum() / ss ** 2
    v_offset_fam = cr1 * (s_of ** 2).sum() / n ** 2
    rows.append(dict(order=o, n=int(n), families=F, slope=slope, offset=offset,
                     v_slope=v_slope, v_offset=v_offset,
                     v_slope_fam=v_slope_fam, v_offset_fam=v_offset_fam))

c_mean = np.mean([r['offset'] for r in rows])
print(f"{'order':14s} {'n':>4s} {'fam':>4s}  {'exponent (indep.)':>19s}  {'exponent (clustered)':>21s}  {'offset (indep.)':>17s}  {'offset (clustered)':>19s}")
for r in rows:
    print(f"{r['order']:14s} {r['n']:4d} {r['families']:4d}  "
          f"{r['slope']:6.3f} +/- {Z*np.sqrt(r['v_slope']):.3f}      "
          f"{r['slope']:6.3f} +/- {Z*np.sqrt(r['v_slope_fam']):.3f}      "
          f"{r['offset']-c_mean:+6.3f} +/- {Z*np.sqrt(r['v_offset']):.3f}    "
          f"{r['offset']-c_mean:+6.3f} +/- {Z*np.sqrt(r['v_offset_fam']):.3f}")
print('(offsets relative to their mean; errors are 2 standard errors)')

## Do the orders differ?

The excess-scatter test of SM Sec. E: with estimates $\hat\theta_k$ and noise variances $v_k$, $Q=\sum_k(\hat\theta_k-\bar\theta_w)^2/v_k$ is $\chi^2_{m-1}$ if all orders share one value, the score $[Q-(m-1)]/\sqrt{2(m-1)}$ is a standard normal deviate, and $\hat\sigma^2=[Q-(m-1)]/[\sum w_k-\sum w_k^2/\sum w_k]$ (with $w_k=1/v_k$) is the spread that reproduces $Q$. The resolution limit for a spread is $\sigma_{\min}^2 = 2\sqrt2\,v/\sqrt m$ with $v$ the typical noise variance (SM Eq. E7); a spread below it cannot be told from noise at this $m$.

In [ ]:
def excess_scatter(est, v):
    est, v = np.asarray(est), np.asarray(v)
    m = len(est)
    w = 1.0 / v
    mean_w = (w * est).sum() / w.sum()
    q = (w * (est - mean_w) ** 2).sum()
    score = (q - (m - 1)) / np.sqrt(2 * (m - 1))
    sigma_hat = np.sqrt(max(0.0, (q - (m - 1)) / (w.sum() - (w ** 2).sum() / w.sum())))
    sigma_min = np.sqrt(Z * np.sqrt(2.0 / m) * np.median(v))
    return score, sigma_hat, sigma_min

for label, ks, ko in [('independent residuals', 'v_slope', 'v_offset'),
                      ('clustered by family', 'v_slope_fam', 'v_offset_fam')]:
    print(label)
    for what, key, vkey in [('exponent', 'slope', ks), ('prefactor', 'offset', ko)]:
        score, s_hat, s_min = excess_scatter([r[key] for r in rows], [r[vkey] for r in rows])
        verdict = 'subgroups resolved' if s_hat > s_min else 'not resolvable'
        print(f'  {what:9s}: score {score:5.1f}, spread {s_hat:.3f} vs limit {s_min:.3f}  -> {verdict}')

With independent residuals both the prefactors and the exponents look heterogeneous. Clustering by family leaves the prefactor result intact but brings the exponent spread down to the resolution limit, so a single exponent cannot be rejected: prefactor subgroups resolve, exponent subgroups do not, as the paper reports.